## POD Line Delta SDI Metrics
This script analyzes full-length and segmented POD lines to assess Delta SDI metrics, integrating treatment-specific data for wildfire suppression analysis.

For segmented POD lines, the script divides them into smaller sections to capture variations in suppression difficulty. It generates points along the lines, splits them at set intervals, and buffers the segments. Zonal statistics are then applied to both segmented and full-length POD lines to calculate mean and median Delta SDI values based on specified fuel treatments. Additionally, the script computes percent change in SDI by comparing Delta SDI rasters to baseline values, expressing differences as a percentage.

For full-length POD lines, the script processes them without segmentation to assess overall treatment effects. This structured workflow ensures a comprehensive analysis of how fuel treatments influence suppression difficulty across different POD features.

#### Import libraries

In [ ]:
import arcpy, os, sys
from arcpy.sa import *

#### Calling in datasets Create variables
Import the necessary datasets for processing the Delta SDI data. This includes the file path to the POD lines shapefile or feature class, as well as the list of Delta SDI rasters that are of interest for the analysis. 

In [ ]:
#input the file path for the POD lines shapefile or feautre class for .gdb
InPOD = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\Bighorn\Bighorn_PODs\Bighorn_PODs_lines.shp'

#define the file path and list of delta SDI rasters of interest for processing
delta_path = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\Bighorn\SDI\Delta_SDI' #file path to delt SDI rasters
delta_trts = ['hand_thin_delta_sdi_Pct90_Bighorn', 'mast_delta_sdi_Pct90_Bighorn', 'mech_thin_delta_sdi_Pct90_Bighorn', 'patch_cut_sm_delta_sdi_Pct90_Bighorn'] #list of delta SDI rasters of interest NOTE these need to be named like this "treatment_type_delta_sdi_Pct#_Name"
delta_files = [os.path.join(delta_path, f'{name}.tif') for name in delta_trts] #join file path and raster names to create a complete file path to each raster in list

#input filepath to the baseline SDI tif
baseline_sdi = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\Bighorn\SDI\SDI_Calc\Outputs\baseline_sdi_Pct90_Bighorn.tif'

#input buffer distance
buffer_distance = '500 Feet'

#this script generates multiple intermediate outputs
#set the option below to "Yes" or "No" to decide whether to delete them after execution
delete_scratch_gdb = 'Yes' 

#### Set environments

In [ ]:
from arcpy import env #setting the workspace and environment
workspace = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\Bighorn\SDI' #change this to match your directory structure this refers to the SDI folder created by the Delt SDI script

#set up scratch workspace 
#the necessary folders and .gdb file will be created in the folder structure setup section below
arcpy.env.workspace = workspace + r'\POD_Line_Delta_SDI\Scratch.gdb' 

#input spatial reference
#NAD83 UTM Zone12 = 26912|NAD83 UTM Zone13 = 26913|North American Albers Equal Area Conic = 102008
env.outputCoordinateSystem = arcpy.SpatialReference(26913) #set spatial reference 

env.overwriteOutput = True #sets outputs to overwrite

arcpy.env.addOutputsToMap = False #stops outputs from being displayed in TOC

#### Set up folder structure
Create folders to organize the data prepared by this script and maintain a standardized folder structure, which will facilitate integration of the Delta SDI script. This process also includes establishing a scratch and final .gdb designed for the intermediate data generated during the POD line segmentation analysis.

In [ ]:
#create the POD_Line_Delta_SDI folder as the root directory for all output data data related to analysis
delta_sdi_lines_folder = os.path.join(workspace, 'POD_Line_Delta_SDI') 
if not os.path.exists(delta_sdi_lines_folder):
    os.makedirs(delta_sdi_lines_folder)

#create Outputs folder to store all POD segmentation and delta SDI data 
outputs_folder = os.path.join(delta_sdi_lines_folder, 'Outputs') 
if not os.path.exists(outputs_folder):
    os.makedirs(outputs_folder)
    
#create a file .gdb file for the scratch workspace to store intermediate data generated during the POD line segmentation analysis
scratch_gdb_path = os.path.join(delta_sdi_lines_folder, 'Scratch.gdb')
if not os.path.exists(scratch_gdb_path):
    arcpy.CreateFileGDB_management(delta_sdi_lines_folder, 'Scratch')
    
#create a file .gdb to store the final outputs for deliverables 
output_gdb = os.path.join(outputs_folder, 'POD_Line_Delta_SDI.gdb')
if not os.path.exists(output_gdb):
    arcpy.CreateFileGDB_management(outputs_folder, 'POD_Line_Delta_SDI')

#### Geospatial Data Processing and Analysis for POD Lines
This section first removes overlapping or messy lines resulting from converting POD polygons to lines by dissolving the lines. Next, it buffers the POD lines at specified distance at entire POD length. Then it segments the POD lines at the desired distance by placing points along the lines and splitting them at those points. It then buffers each resulting segment to create a defined spatial area around the segments, supporting the analysis to obtain Delta SDI and SDI metrics.

In [ ]:
#clean buffer and segmentment POD lines
#dissolve POD lines to remove overlaps and clean up messy line segments  
#this process creates lines that are split at intersections, preserving full POD line lengths  
print('Dissolve Lines to Remove Overlapping Lines')
POD_single_part = arcpy.management.Dissolve(InPOD, 'POD_Line_Dissolve', None, None, "SINGLE_PART", "DISSOLVE_LINES", "")

print('Buffer Full POD Length')
#buffer full length POD lines at specified distance
POD_Line_Buffer = arcpy.analysis.PairwiseBuffer(POD_single_part, 'POD_Line_Buffer', buffer_distance) 

#place points along single feature POD lines at specified distance
print('Generate Points Along Lines')
PointsAlongLines = arcpy.management.GeneratePointsAlongLines(POD_single_part, 'POD_Seg_PointsAlongLines', Point_Placement='DISTANCE',
                                                             Distance='2 Kilometers') #change this value for different segment lengths

#split POD lines at points of specified distance
print('Split Line At Points')
POD_Split = arcpy.management.SplitLineAtPoint(POD_single_part, PointsAlongLines, 'POD_Seg_Split', '5 Meters')
#NOTE: the segmented POD lines may be split inaccurately or into smaller, incorrect features when using these tools (Generate Points Along Lines and/or Split Line At Point)
#this issue is addressed when joining the data back into the lines in a later step

#buffer splited POD lines at specified distance
print('Buffer Split Lines')
POD_Buffer = arcpy.analysis.PairwiseBuffer(POD_Split, 'POD_Seg_Buffer', buffer_distance) 

print('\nDone cleaning, buffering, and segmentmenting POD lines')

#### Cleaning buffered outputs
This section of the code addresses small or inaccurate features created during the buffering processing for both the segmented POD lines and full length POD lines. It begins by removing overlapping features using the RemoveOverlapMultiple tool to eliminate overlaps between polygons. However, this process can result in small or inaccurate features, as well as features with NULL values, which are removed and merged with neighboring larger polygons. The script then enters an iterative process where it continuously calculates the acres field for each polygon and selects those with an area smaller than 70 acres. These small polygons are eliminated by merging them with neighboring larger polygons using the Eliminate tool. The loop continues until no polygons with an area smaller than 70 acres remain. The threshold of 70 acres was chosen for its effectiveness in removing small, irrelevant features while preserving meaningful polygon sizes for further analysis.

In [ ]:
#define input datasets
#create a list of segmented and full-length POD line outputs 
#names were generated from cleaning, buffering, and segmenting processes
buffered_inputs = ['POD_Seg_Buffer', 'POD_Line_Buffer']
#create list of output names 
output_names = ['POD_Seg_Buffer_Remove', 'POD_Line_Buffer_Remove']

#loop through buffered inputs and assign each to its corresponding output name
for buffered_input, output_name in zip(buffered_inputs, output_names):
    print(f'Cleaning Buffered Lines for {output_name}\n')
    #after buffering, there are overlapping ends to every feature
    #to address this there are a few steps to take
    print('Remove Overlapping Features') 
    POD_Remove = arcpy.analysis.RemoveOverlapMultiple(buffered_input,  output_name, 'CENTER_LINE', 'ALL')
    #this tool may generate small or inaccurate features, especially in areas with numerous overlapping features, 
    #and it can also produce polygons with NULL values
    #these polygons should be removed and merged with adjacent larger polygons
    #the following steps will address this issue
    
    print('Cleaning Overlapping Features')
    #remove any NULL values that may have been created during the remove overlapping features process
    #create and calculate acres field
    print('Add and Calculate Acres Field')
    arcpy.management.CalculateGeometryAttributes(POD_Remove, 'acres AREA', '', 'ACRES_US', '', "SAME_AS_INPUT")
        
    #delete features with NULL acres
    print('Delete Features with NULL Acres')
    POD_Remove_Select = arcpy.management.SelectLayerByAttribute(POD_Remove, 'NEW_SELECTION', '"acres" IS NULL')
        
    #check if any features are selected with NULL acres
    count = int(arcpy.management.GetCount(POD_Remove_Select)[0])
    if count > 0:
        #if there are any selected features with NULL acres, the DeleteFeatures tool will be used to remove them
        arcpy.management.DeleteFeatures(POD_Remove_Select)
        #print the number of deleted features
        print(f'Deleted {count} features with NULL acres')
    else:
        #if no features were found with NULL acres, print a message indicating that no deletions were necessary
        print('No features with NULL acres found')
    
    
    
    #remove polygons smaller than 70 acres that may have been created during the buffering or remove overlapping features process,
    #by iteratively eliminating small polygons until none remain
    
    #initialize iteration counter
    #this counter is used to track the number of iterations in the loop and to generate unique output names
    iteration = 0  
    #iteratively remove polygons smaller than 70 acres until no such polygons remain
    while True:
        #calculate acres field
        print('Calculate Acres Field')
        arcpy.management.CalculateGeometryAttributes(POD_Remove, 'acres AREA', '', 'ACRES_US', '', "SAME_AS_INPUT")
    
        #select all polygons with acres less than 70
        print('Select By Attribute')
        POD_Remove_Select = arcpy.management.SelectLayerByAttribute(POD_Remove, 'NEW_SELECTION', '"acres" < 70')
        
        #check if any features are selected
        count = int(arcpy.management.GetCount(POD_Remove_Select)[0])
        if count == 0: #if the count is zero, indicating no polygons with acres < 70 remain
            print('No more polygons with acres less than 70')
            break  #exit the loop if no features are selected
    
        print(f'{count} polygons with acres < 70 selected. Eliminating...')
        
        #create a unique output name for the eliminated layer the output feature class cannot be the same as input input layer 
        #the output name incorporates the iteration number to the unique output name
        output_feature_class = f'{output_name}_Iter_{iteration}'
        
        #use the Eliminate tool to merge small polygons (those with acres < 70) with neighboring larger polygons
        POD_Remove = arcpy.management.Eliminate(POD_Remove_Select, output_feature_class, 'AREA')
        
        #update the iteration counter to track the next elimination step
        iteration += 1

    #assign the final cleaned dataset to its respective variable
    if output_name == "POD_Seg_Buffer_Remove":
        POD_Seg_Buffer_Remove = POD_Remove
    elif output_name == "POD_Line_Buffer_Remove":
        POD_Line_Buffer_Remove = POD_Remove
    
    print(f'Done cleaning lines for {output_name}\n')
print('Cleaning Process Completed')

#### Zonal statistics\table field management for Segmented delta SDI 
This section of the code processes the Delta SDI simulated fuel treatments by performing zonal statistics, dynamically managing field names, and calculating percent change in SDI values. It begins by iterating through each Delta SDI raster, using the Zonal Statistics As Table tool to compute the mean and median values for each POD line segment. To ensure organized output, the base filename of each raster is extracted and modified to create dynamic names for both the zonal statistics tables and corresponding field names. The script then renames the mean and median fields to reflect the specific treatment applied, such as “mean_handthindeltasdiPct90.” These newly created fields are then joined to the corresponding feature class, and the updated feature class is saved to the final output location. 

Additionally, the script calculates the percent change in SDI values by taking the absolute difference between the Delta SDI raster and the baseline SDI, then dividing by the baseline SDI to express the difference as a percentage. This percent change raster is saved and analyzed using another round of zonal statistics, extracting the mean percent change for each POD line segment. The resulting percent change field, such as “pct_meanhandthindeltasdiPct90,” is renamed and joined back to the feature class.

In [ ]:
#running zonal statistics and table field management for segmented delta SDI simulated fuel treatments

#initialize an empty list to store the output field names, 
#which will be used as input fields for the spatial join to join back with the POD lines
output_names = []

#loop through delta sdi simulated fuel treatments and use Zonal Statistics As Table to append stat values to the feature class                                          
for delta_file in delta_files:
    #NOTE: delta SDI rasters must follow the naming convention "treatment_type_delta_sdi_Pct#_Name"
    #this format is essential for ensuring the correct dynamic naming
    
    #construct unique name by modifying the extracted base raster name
    base_name = os.path.basename(delta_file) #extract the base part of the filename
    base_name = os.path.basename(delta_file).replace('.tif', '') #removing the .tif extension
    print(f'Perform Zonal Statistics on segmented {base_name}...')
    parts = base_name.split('_') #split the name by underscores
    before_sdi = '_'.join(parts[:parts.index('sdi')]) #extract everything before 'sdi' in the name
    pct_part = [part for part in parts if part.startswith('Pct')][0] #identify the part of the name that starts with 'Pct'
    pct_number = pct_part.replace('Pct', '') #remove 'Pct' and keep the number
    
    #combine the extracted parts to form the Zonal Statistics table and field name
    output_name = f"{before_sdi}_sdi{pct_number}"

    #construct field name by removing underscores
    field_name = output_name.replace("_", "")
    
    #run Zonal Statistics to calculate the mean and median values of specific delta SDI simulated fuel treatment for each POD line segment
    #create the Zonal Statistics table name
    output_table_name = f'ZonalStat_Seg_{output_name}'  #add ZonalStat_ prefix to the output name
    output_table = os.path.join(env.workspace, output_table_name) #create output location, even though its the same 
                                                                  #as the env.workspace the tool wouldn't work 
                                                                  #without setting this
    #run the Zonal Statistics As Table with ALL statistics ensures both MEAN and MEDIAN are calculated
    POD_Zonal_Stat = ZonalStatisticsAsTable(POD_Seg_Buffer_Remove, 'ORIG_FID', delta_file, output_table, 'DATA', 'ALL') 
    
    #manage table fields by changing the field names to reflect the specific treatment applied
    #define the original field names ('MEAN' and 'MEDIAN') which are the default outputs from the Zonal Statistics tool
    #and create new field names dynamically based on the treatment name
    old_mean_field = 'MEAN'  #default mean field from Zonal Statistics
    old_median_field = 'MEDIAN'  #default median field from Zonal Statistics
    new_mean_field = f'mean_{field_name}'  #construct new mean field name
    new_median_field = f'median_{field_name}'  #construct new median field name

    #rename the mean and median fields in the Zonal Statistics table using AlterField to reflect the specific treatment applied
    arcpy.AlterField_management(POD_Zonal_Stat, old_mean_field, new_mean_field, new_mean_field)
    arcpy.AlterField_management(POD_Zonal_Stat, old_median_field, new_median_field, new_median_field)

    #join the mean and median fields to corresponding feature class 
    arcpy.management.JoinField(POD_Seg_Buffer_Remove, 'ORIG_FID', POD_Zonal_Stat, 'ORIG_FID', [new_mean_field, new_median_field])
    
    #append the field names to the output_names list for later use in joining the calculated values back to the POD lines
    output_names.append(new_mean_field)
    output_names.append(new_median_field)

    print(f'Calculate percent change for segmented {base_name}...')
    #calculate percent change in SDI values using Raster Calculator
    #this computes the absolute difference between the delta SDI raster and the baseline SDI
    #then divides by the baseline SDI to express the difference as a percentage
    percent_change_raster = (arcpy.sa.Abs(arcpy.sa.Raster(delta_file)) / arcpy.sa.Raster(baseline_sdi)) * 100
    
    #save the computed percent change raster directly
    percent_change_raster.save(f'percent_{output_name}')  

    print(f'Perform Zonal Statistics on segmented percent change for {base_name}...\n')
    #run Zonal Statistics to calculate the mean of percent change in SDI values of specific delta SDI simulated fuel treatment for each POD line segment
    #create the Zonal Statistics table name
    percent_output_table_name = f'ZonalStat_Seg_Percent_{output_name}'
    percent_output_table = os.path.join(env.workspace, percent_output_table_name)#create output location, even though its the same 
                                                                                 #as the env.workspace the tool wouldn't work 
                                                                                 #without setting this

    #run the Zonal Statistics As Table to calculates the MEAN percent change in SDI
    Percent_Zonal_Stat = ZonalStatisticsAsTable(POD_Seg_Buffer_Remove,  'ORIG_FID', percent_change_raster, percent_output_table, 'DATA', 'MEAN')  

    #manage table fields by changing the field names to reflect the specific treatment applied
    #define the original field names ('MEAN') which is the default output from the Zonal Statistics tool
    #and create new field name for the mean percent change value dynamically based on the treatment name
    mean_percent_field = f'pct_mean_{field_name}'
    
    #rename the mean field in the Zonal Statistics table using AlterField to reflect the specific treatment applied
    arcpy.AlterField_management(Percent_Zonal_Stat, 'MEAN', mean_percent_field, mean_percent_field)
    
    #join the mean and median fields to corresponding feature class
    arcpy.management.JoinField(POD_Seg_Buffer_Remove, 'ORIG_FID', Percent_Zonal_Stat, 'ORIG_FID', mean_percent_field)
    
    #append the field names to the output_names list for later use in joining the calculated values back to the POD lines
    output_names.append(mean_percent_field)

print('Cleaning table')
#clena and remove unwanted fields created during the creation of the onal statistics tables and joins
#retrieve all field names from the POD feature class
all_fields = [field.name for field in arcpy.ListFields(POD_Seg_Buffer_Remove)]

#identify fields that contain "ORIG" in their names
#these fields may be remnants created during the creation of the onal statistics tables and joins and should be removed
fields_to_remove = [field for field in all_fields if 'ORIG' in field.upper()]

#additionally check if the field "BUFF_DIST" exists and add it to the removal list
if 'BUFF_DIST' in all_fields:
    fields_to_remove.append('BUFF_DIST')

#removes unwanted fields if any exist
if fields_to_remove:
    arcpy.management.DeleteField(POD_Seg_Buffer_Remove, fields_to_remove)
    print(f"Removed fields: {', '.join(fields_to_remove)}")
else:
    print('No unwanted fields found for removal')

print('Join table and save out feature class')

#save the final output feature class containing all calculated zonal statistics and percent change values
#this output represents the final processed version of the dataset with all treatment effects accounted for
final_output = os.path.join(output_gdb, 'POD_Segmented_Delta_SDI_Metrics_Final_Poly')

#copy the processed POD feature class to the specified geodatabase
arcpy.management.CopyFeatures(POD_Seg_Buffer_Remove, final_output)

print('\nAll Done calculating metrics for segmented POD lines')

#### Join delta SDI data back into the segmented lines
This section brings the calculated Delta SDI data back into the POD line data by performing spatial joins and dissolving segmented POD lines to align with the Delta SDI extract polygons. The code joins the Delta SDI data back to the segmented POD lines using the Spatial Join tool. It then dissolves these segmented POD lines, which may have been split inaccurately or into smaller, incorrect features when using tools like Generate Points Along Lines and/or Split Line At Point. Additionally, if the POD lines have slivers or if the lines between PODs are not joined properly (i.e., not flush with one another), this can cause inaccuracies by adding extra length to the segments. Dissolving the segmented lines combines the disconnected lines with all the lines that fall within a calculated buffer area, ensuring that both the final segment polygons and lines align. The dissolve operation groups the lines based on relevant attributes from the Delta SDI data, with dynamically generated fields reflecting the specific treatments and acres applied. The final result is a feature class where the segmented POD lines are saved to the final output location.

In [ ]:
print('Join Delta SDI Metrics Back to Segmented POD Lines')
#using sptail join to join delta SDI data back into the segmented lines
POD_SPJoin = arcpy.analysis.SpatialJoin(POD_Split, POD_Seg_Buffer_Remove, 'POD_Seg_SpatialJoin', 'JOIN_ONE_TO_ONE',
                                        'KEEP_ALL', '', 'LARGEST_OVERLAP', None, '', None)
print('Dissolve Segmented POD Lines')
#the Generate Points Along Lines and/or Split Line At Point tools may have split POD lines into smaller or inaccurate features
#the dissolve tool is used to combine disconnected lines with all the lines that fall within each segmented polygon
#based on fields derived from the Delta SDI extract polygons, including the dynamically generated output names that reflect the specific treatments

#use the dynamically generated output names
dissolve_fields = 'acres;' + ';'.join(output_names) #create a string of field names for dissolve
 
#specify final output .gdb location and name
final_output = os.path.join(output_gdb, 'POD_Segmented_Delta_SDI_Metrics_Final_Line')
#run the Pairwise Dissolve tool with the dynamically generated fields
POD_Dissolve = arcpy.analysis.PairwiseDissolve(POD_SPJoin, final_output, dissolve_fields, None, 'MULTI_PART', '')

print('\nAll Done with Join')

#### Zonal statistics\table field management for full length POD line delta SDI 
This section of the code processes the Delta SDI simulated fuel treatments by performing zonal statistics, dynamically managing field names, and calculating percent change in SDI values for full-length POD lines. The process begins by retrieving previously saved percent change rasters from the scratch geodatabase to avoid unnecessary re-creation. It then iterates through each Delta SDI raster, using the Zonal Statistics As Table tool to compute the mean and median values for each full-length POD line. To maintain organized outputs, the base filename of each raster is extracted and modified, creating dynamic names for the zonal statistics tables and corresponding field names. The script renames the mean and median fields to reflect the specific treatment applied, such as “mean_hand_thin_delta_sdi_Pct90.” These newly created fields are then joined to the corresponding feature class, and the updated feature class is saved to the final output location.

In addition to standard zonal statistics, the script processes percent change in SDI values. It iterates through the previously saved percent change rasters and calculates zonal statistics on these datasets to determine the mean percent change for each POD line. The extracted field, such as “pct_mean_hand_thin_delta_sdi_Pct90,” is dynamically renamed and joined to the feature class, ensuring that both absolute and relative treatment effects are captured.

In [ ]:
#running zonal statistics and table field management for full length POD lines delta SDI simulated fuel treatments

#retrieve all previously saved percent change rasters accesses the scratch .gdb to list existing percent change rasters  
#avoids unnecessary re-creation for this analysis 
percent_rasters = [os.path.join(arcpy.env.workspace, raster) for raster in arcpy.ListRasters('percent_*')]

#initialize an empty list to store the output field names, 
#which will be used as input fields for the spatial join to join back with the POD lines
output_names_lines = []

#loop through delta sdi simulated fuel treatments and use Zonal Statistics As Table to append stat values to the feature class
for delta_file in delta_files:
    #construct unique name by modifying the extracted base raster name
    base_name = os.path.basename(delta_file) #extract the base part of the filename
    base_name = os.path.basename(delta_file).replace('.tif', '') #removing the .tif extension
    print(f'Perform Zonal Statistics on POD line {base_name}...')
    parts = base_name.split('_') #split the name by underscores
    before_sdi = '_'.join(parts[:parts.index('sdi')]) #extract everything before 'sdi' in the name
    pct_part = [part for part in parts if part.startswith('Pct')][0] #identify the part of the name that starts with 'Pct'
    pct_number = pct_part.replace('Pct', '') #remove 'Pct' and keep the number 

    #combine the extracted parts to form the Zonal Statistics table and field name
    output_name = f'{before_sdi}_sdi{pct_number}'

    #construct field name by removing underscores
    field_name = output_name.replace('_', '')

    #run Zonal Statistics to calculate the mean and median values of specific delta SDI simulated fuel treatment for each POD line segment
    #create the Zonal Statistics table name
    output_table_name = f'ZonalStat_Line_{output_name}' #add ZonalStat_ prefix to the output name
    output_table = os.path.join(arcpy.env.workspace, output_table_name) #create output location, even though its the same 
                                                                        #as the env.workspace the tool wouldn't work
                                                                        #without setting this
    #run the Zonal Statistics As Table with ALL statistics ensures both MEAN and MEDIAN are calculated
    POD_Zonal_Stat = arcpy.sa.ZonalStatisticsAsTable(POD_Line_Buffer_Remove, 'ORIG_FID', delta_file, output_table, 'DATA', 'ALL')

    #manage table fields by changing the field names to reflect the specific treatment applied
    #define the original field names ('MEAN' and 'MEDIAN') which are the default outputs from the Zonal Statistics tool
    #and create new field names dynamically based on the treatment name
    old_mean_field = 'MEAN'  #default mean field from Zonal Statistics
    old_median_field = 'MEDIAN'  #default median field from Zonal Statistics
    new_mean_field = f'mean_{field_name}'  #construct new mean field name
    new_median_field = f'median_{field_name}'  #construct new median field name

    #rename the mean and median fields in the Zonal Statistics table using AlterField to reflect the specific treatment applied
    arcpy.AlterField_management(POD_Zonal_Stat, old_mean_field, new_mean_field, new_mean_field)
    arcpy.AlterField_management(POD_Zonal_Stat, old_median_field, new_median_field, new_median_field)

    #join the mean and median fields to corresponding feature class
    arcpy.management.JoinField(POD_Line_Buffer_Remove, 'ORIG_FID', POD_Zonal_Stat, 'ORIG_FID', [new_mean_field, new_median_field])

    #append the field names to the output_names list for later use in joining the calculated values back to the POD lines
    output_names_lines.append(new_mean_field)
    output_names_lines.append(new_median_field)
    
print('Done calculating Zonal Statistics on POD lines\n')

#run zonal statistics and table field management to process percent change for full length POD line
#loop percent change rasters and use Zonal Statistics As Table to append stat values to the feature class
for percent_raster in percent_rasters:
    #construct unique name by modifying the extracted base raster name
    base_name = os.path.basename(percent_raster) #extract the base part of the filename
    base_name = os.path.basename(percent_raster).replace('.tif', '') #removing the .tif extension
    print(f'Perform Zonal Statistics on POD line percent change for {base_name}...')
    clean_name = base_name.replace('percent_', '')  #eemove 'percent_' prefix
    
    #construct field name by removing underscores
    field_name = clean_name.replace('_', '')
    
    #run Zonal Statistics to calculate the mean of percent change in SDI values of specific delta SDI simulated fuel treatment for each POD line segment
    #create the Zonal Statistics table name
    output_table_name = f'ZonalStat_Lines_Percent_{base_name}'
    output_table = os.path.join(arcpy.env.workspace, output_table_name)#create output location, even though its the same 
                                                                       #as the env.workspace the tool wouldn't work 
                                                                       #without setting this

    #run the Zonal Statistics As Table to calculates the MEAN percent change in SDI
    Percent_Zonal_Stat_Lines = ZonalStatisticsAsTable(POD_Line_Buffer_Remove,  'ORIG_FID', percent_raster, output_table, 'DATA', 'MEAN') 

    #manage table fields by changing the field names to reflect the specific treatment applied
    #define the original field names ('MEAN') which is the default output from the Zonal Statistics tool
    #and create new field name for the mean percent change value dynamically based on the treatment name
    mean_percent_field_lines = f'pct_mean_{field_name}'

    #rename the mean field in the Zonal Statistics table using AlterField to reflect the specific treatment applied
    arcpy.AlterField_management(Percent_Zonal_Stat_Lines, 'MEAN', mean_percent_field_lines, mean_percent_field_lines)

    #join the mean and median fields to corresponding feature class
    arcpy.management.JoinField(POD_Line_Buffer_Remove, 'ORIG_FID', Percent_Zonal_Stat_Lines, 'ORIG_FID', mean_percent_field_lines)

    #append the field names to the output_names list for later use in joining the calculated values back to the POD lines
    output_names_lines.append(mean_percent_field_lines)
    
print('Done calculating Zonal Statistics on POD line percent change\n')

print('Cleaning table')
#clena and remove unwanted fields created during the creation of the onal statistics tables and joins
#retrieve all field names from the POD feature class
all_fields = [field.name for field in arcpy.ListFields(POD_Line_Buffer_Remove)]

#identify fields that contain "ORIG" in their names
#these fields may be remnants created during the creation of the onal statistics tables and joins and should be removed
fields_to_remove = [field for field in all_fields if 'ORIG' in field.upper()]

#additionally check if the field "BUFF_DIST" exists and add it to the removal list
if 'BUFF_DIST' in all_fields:
    fields_to_remove.append('BUFF_DIST')
    
#removes unwanted fields if any exist
if fields_to_remove:
    arcpy.management.DeleteField(POD_Line_Buffer_Remove, fields_to_remove)
    print(f"Removed fields from POD lines: {', '.join(fields_to_remove)}")
else:
    print('No unwanted fields found for removal in POD lines')

print('Join table and save out feature class')

#save the final output feature class containing all calculated zonal statistics and percent change values
#this output represents the final processed version of the dataset with all treatment effects accounted for
final_line_output = os.path.join(output_gdb, 'POD_Line_Delta_SDI_Metrics_Final_Poly')

#copy the processed POD feature class to the specified geodatabase
arcpy.management.CopyFeatures(POD_Line_Buffer_Remove, final_line_output)

print('\nAll Done calculating metrics for POD lines')

#### Join delta SDI data back into the full length POD lines
The process begins with a spatial join, where Delta SDI metrics are assigned to the POD lines using the largest overlap method, ensuring accurate integration of treatment-specific data. The dissolve operation then merges any potentially disconnected line portions while preserving dynamically generated fields, including treatment-specific metrics and total acres applied. The final output is a feature class containing the full-length POD lines with integrated Delta SDI attributes, saved to the designated geodatabase for further analysis.

In [ ]:
print('Join Delta SDI Metrics Back to POD Lines')
#using sptail join to join delta SDI data back into the POD lines
POD_Line_SPJoin = arcpy.analysis.SpatialJoin(POD_single_part, POD_Seg_Buffer_Remove, 'POD_Line_SpatialJoin', 'JOIN_ONE_TO_ONE',
                                        'KEEP_ALL', '', 'LARGEST_OVERLAP', None, '', None)
print('Dissolve POD Lines')
#the dissolve tool is used to combine any poteinal disconnected lines
#based on fields derived from the Delta SDI extract polygons, including the dynamically generated output names that reflect the specific treatments

#use the dynamically generated output names
dissolve_fields = 'acres;' + ';'.join(output_names_lines) #create a string of field names for dissolve
 
#specify final output .gdb location and name
final_output = os.path.join(output_gdb, 'POD_Line_Delta_SDI_Metrics_Final_Line')
#run the Pairwise Dissolve tool with the dynamically generated fields
POD_Dissolve = arcpy.analysis.PairwiseDissolve(POD_Line_SPJoin, final_output, dissolve_fields, None, 'MULTI_PART', '')

print('\nAll Done with Join')

#### Extracting delta SDI raster to buffered POD lines
This section extracts Delta SDI raster data to the buffered POD lines, primarily for visualization purposes. The code loops through each Delta SDI raster file, constructs a unique output name by modifying the raster’s base name, and runs the Extract By Mask tool to extract the raster data for the areas within the buffered POD lines. The final output is saved to the final output location. 

In [ ]:
#extract delta SDI data to buffered POD lines to support with visual ads
print('Extract Delta SDI Rasters to the Buffered POD Lines')
#loop through Delta SDI Treatment Rasters
for delta_file in delta_files:
    
    #construct unique name by modifying the extracted base raster name
    base_name = os.path.basename(delta_file) #extract the base part of the filename    
    base_name = os.path.basename(delta_file).replace('.tif', '')  #removing the .tif extension
    print('Perform extraction on ' + base_name + '...') 
    parts = base_name.split('_')  #split the name by underscores
    before_sdi = '_'.join(parts[:parts.index('sdi')])  #extract everything before 'sdi' in the name
    pct_part = [part for part in parts if part.startswith('Pct')][0]  #identify the part of the name that starts with 'Pct'
    pct_number = pct_part.replace('Pct', '')  #remove 'Pct' and keep the number
    
    #combine the extracted parts to form the Zonal Statistics table and field name
    output_name = f'{before_sdi}_sdi{pct_number}_extract'
        
    #run extract by mask tool on the delta sdi raster on the buffered POD lines
    #specify finale output .gdb location and name
    final_output = os.path.join(output_gdb, output_name)
    #run Extract By Mask tool 
    extract = arcpy.sa.ExtractByMask(delta_file, POD_Line_Buffer, 'INSIDE')
    #save final output
    extract.save(final_output)
    
print('\nAll Done Segmenting POD Lines and Calculating Delta SDI Metrics')

In [ ]:
#this script produces several intermediate outputs during execution
#the user can choose whether to delete the temporary geodatabase (Scratch.gdb)
#after the script runs to conserve storage and maintain a tidy workspace

#extracts only the file name from the full geodatabase path 
scratch_gdb_name = os.path.basename(scratch_gdb_path)

#check if the user has chosen to delete the scratch geodatabase  
#the value of 'delete_scratch_gdb' should be set to either 'Yes' or 'No'  
#if 'Yes' the script will attempt to delete the geodatabase  
if delete_scratch_gdb == 'Yes':
    #verify if the specified geodatabase actually exists before attempting deletion
    if arcpy.Exists(scratch_gdb_path):
        #delete the geodatabase
        arcpy.Delete_management(scratch_gdb_path)
        print(f"Deleted: {scratch_gdb_name}")
    else:
        print(f"Geodatabase not found: {scratch_gdb_name}")
else:
    print(f"Geodatabase kept: {scratch_gdb_name}")
